# Perbandingan 1D-CNN vs ARIMA untuk Peramalan Revenue Mingguan (Dataset M5)

**Tujuan notebook ini:** membandingkan performa model **1D-CNN** dan **ARIMA** dalam meramalkan
revenue mingguan per kombinasi *state* x *kategori* pada dataset M5 Forecasting Accuracy,
menggunakan skema evaluasi **walk-forward** yang **identik** untuk kedua model agar perbandingannya adil (apple-to-apple).

**Alur notebook:**
1. Unduh & muat dataset M5
2. Praproses data (filter, gabung kalender & harga, hitung revenue)
3. Agregasi ke level mingguan per *state* x *kategori*
4. Konfigurasi eksperimen
5. Tuning hyperparameter 1D-CNN (Keras Tuner) dan ARIMA (grid search AIC)
6. Evaluasi walk-forward yang adil untuk kedua model
7. Tabel & visualisasi perbandingan akhir

> **Catatan performa:** karena kedua model di-*retrain ulang* di setiap langkah walk-forward (bukan cuma sekali),
> proses ini cukup lambat. Untuk uji coba cepat, kecilkan `TEST_WEEKS` dan/atau `CNN_EPOCHS` di sel konfigurasi.


## 1. Import Library

In [ ]:
import os
import gc
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


## 2. Unduh Dataset M5

Dataset diunduh dari Kaggle (`aryayadav0513/m5-forecasting-accuracy`) menggunakan `kagglehub`.
File yang dipakai: `sales_train_validation.csv`, `calendar.csv`, `sell_prices.csv`.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("aryayadav0513/m5-forecasting-accuracy")
dataset_path = os.path.join(path, "m5-forecasting-accuracy")
print("Path dataset:", dataset_path)
print("Isi folder :", os.listdir(dataset_path))


## 3. Muat & Praproses Data

Langkah-langkah:
1. Muat `sales_train_validation.csv`, `calendar.csv`, `sell_prices.csv`.
2. Filter hanya kombinasi *state* (`CA`, `TX`, `WI`) dan *kategori* (`HOBBIES`, `HOUSEHOLD`, `FOODS`) yang dipakai di eksperimen ini.
3. Ambil sampel 50% item dan separuh kolom hari (`d_*`) pertama agar ukuran data lebih ringan untuk eksperimen.
4. Ubah ke format *long* (`melt`), lalu gabungkan dengan `calendar` (untuk `wm_yr_wk`) dan `sell_prices` (untuk harga jual).


In [ ]:
sales = pd.read_csv(os.path.join(dataset_path, "sales_train_validation.csv"))
calendar = pd.read_csv(os.path.join(dataset_path, "calendar.csv"))
prices = pd.read_csv(os.path.join(dataset_path, "sell_prices.csv"))

print("sales   :", sales.shape)
print("calendar:", calendar.shape)
print("prices  :", prices.shape)


In [ ]:
STATES = ['CA', 'TX', 'WI']
CATEGORIES = ['HOBBIES', 'HOUSEHOLD', 'FOODS']

# Filter state & kategori yang relevan, lalu ambil sampel 50% item untuk mempercepat eksperimen
sales = sales[
    sales['state_id'].isin(STATES) &
    sales['cat_id'].isin(CATEGORIES)
].sample(frac=0.5, random_state=SEED)

# Ambil separuh kolom hari pertama (mengurangi beban komputasi)
day_cols = [c for c in sales.columns if c.startswith('d_')]
half_days = day_cols[: len(day_cols) // 2]
sales = sales[['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'] + half_days]

print("Bentuk data setelah filter & sampling:", sales.shape)


In [ ]:
# Ubah ke format long: satu baris = satu item pada satu hari
sales_long = sales.melt(
    id_vars=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    var_name='d',
    value_name='sales'
)

# Gabungkan dengan calendar (untuk minggu / wm_yr_wk) dan sell_prices (untuk harga)
sales_long = sales_long.merge(calendar[['d', 'wm_yr_wk']], on='d', how='left')
sales_long = sales_long.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

print("sales_long:", sales_long.shape)
sales_long.head()


## 4. Agregasi Mingguan per State x Kategori (`weekly_avg_enriched`)

Data harian per-item diringkas menjadi **total revenue mingguan** untuk tiap kombinasi *state* x *kategori*
(`revenue = sales x sell_price`, dijumlahkan per minggu), lalu diperkaya dengan fitur kalender `wday` dan `month`.
Fitur-fitur ini yang nantinya jadi input model 1D-CNN dan target peramalan ARIMA.


In [ ]:
sl = sales_long.copy()
sl['sales'] = pd.to_numeric(sl['sales'], errors='coerce').fillna(0)
sl['sell_price'] = pd.to_numeric(sl['sell_price'], errors='coerce')
sl['revenue_row'] = sl['sales'] * sl['sell_price'].fillna(0)

weekly_avg = (
    sl.groupby(['wm_yr_wk', 'state_id', 'cat_id'], as_index=False)
      .agg(revenue=('revenue_row', 'sum'))
)

calendar_features = calendar[['wm_yr_wk', 'wday', 'month']].drop_duplicates('wm_yr_wk')
weekly_avg_enriched = (
    weekly_avg.merge(calendar_features, on='wm_yr_wk', how='left')
              .sort_values(['state_id', 'cat_id', 'wm_yr_wk'])
              .reset_index(drop=True)
)

print("weekly_avg_enriched:", weekly_avg_enriched.shape)
weekly_avg_enriched.head()


## 5. Konfigurasi Eksperimen

Konfigurasi ini dipakai **sama persis** untuk 1D-CNN dan ARIMA supaya perbandingan adil.

In [ ]:
TIME_STEP    = 10   # panjang jendela input (minggu)
TEST_WEEKS   = 30   # panjang periode uji walk-forward (minggu), otomatis menyesuaikan jika data lebih pendek
NUM_FEATURES = 3    # revenue, wday, month
CNN_EPOCHS   = 15   # epoch training CNN di setiap langkah walk-forward (retrain tiap minggu)
CNN_BATCH    = 8

print(f"TIME_STEP={TIME_STEP}, TEST_WEEKS={TEST_WEEKS}, NUM_FEATURES={NUM_FEATURES}, "
      f"CNN_EPOCHS={CNN_EPOCHS}, CNN_BATCH={CNN_BATCH}")


## 6. Fungsi Bantu

- `create_dataset`: mengubah deret waktu menjadi dataset *supervised* dengan jendela geser (*sliding window*) — dipakai untuk menyiapkan input 1D-CNN.
- `wrmsse_score`: menghitung **RMSSE** (RMSE dibagi skala RMS-selisih-pertama dari deret historis) — dipakai sebagai metrik utama, konsisten untuk CNN maupun ARIMA.

In [ ]:
def create_dataset(data, time_step=10, num_features=None):
    """Ubah data time series menjadi dataset supervised dengan fitur lag (sliding window).

    Args:
        data: array (n_samples, n_features).
        time_step: panjang jendela input (jumlah langkah waktu sebelumnya).
        num_features: jumlah fitur yang diambil untuk X (default: semua kolom).

    Returns:
        X, y: fitur (n, time_step, num_features) dan target (n,) -> target = kolom ke-0 (revenue).
    """
    if num_features is None:
        num_features = data.shape[1]
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:(i + time_step), :num_features])
        y.append(data[i + time_step, 0])
    return np.array(X), np.array(y)


def wrmsse_score(y_true, y_pred, scale_series=None):
    """RMSSE: RMSE dibagi skala RMS-selisih-pertama.

    Jika `scale_series` diberikan, skala dihitung dari deret tersebut (mis. histori sebelum
    periode uji) sehingga bisa dibuat identik untuk beberapa model. Jika tidak, skala dihitung
    dari `y_true` itu sendiri.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    base = np.asarray(scale_series) if scale_series is not None else y_true
    diffs = np.diff(base)
    if len(diffs) == 0:
        return rmse
    scale = np.sqrt(np.mean(diffs ** 2))
    return rmse if scale == 0 else rmse / scale


## 7. Tuning Hyperparameter — 1D-CNN (Keras Tuner)

Tuning dilakukan **satu kali** pada deret representatif (`HOBBIES`-`CA`) memakai `RandomSearch` dari Keras Tuner,
lalu hyperparameter terbaik dipakai untuk semua kombinasi *state* x *kategori* pada tahap walk-forward.
**Semua trial** (bukan cuma yang terbaik) dicatat di `cnn_tuning_results` supaya bisa dilampirkan sebagai bukti proses tuning.

In [ ]:
# %pip install -q keras-tuner   # jalankan sekali jika belum terpasang
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

tf.random.set_seed(SEED)


def build_model(hp):
    inputs = Input(shape=(TIME_STEP, NUM_FEATURES))
    x = inputs
    num_cnn_layers = hp.Int('num_cnn_layers', min_value=1, max_value=3, step=1)
    for i in range(num_cnn_layers):
        num_filters = hp.Int(f'filters_{i}', min_value=32, max_value=128, step=32)
        kernel_size = hp.Choice(f'kernel_size_{i}', values=[2, 3, 5])
        x = Conv1D(filters=num_filters, kernel_size=kernel_size,
                   activation='relu', padding='causal')(x)
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate))
    return model


# --- data tuning: deret representatif HOBBIES-CA ---
tuning_series = weekly_avg_enriched[
    (weekly_avg_enriched['cat_id'] == 'HOBBIES') &
    (weekly_avg_enriched['state_id'] == 'CA')
].sort_values('wm_yr_wk')

feat_tune = tuning_series[['revenue', 'wday', 'month']].values
scaler_tune = MinMaxScaler()
feat_tune_scaled = scaler_tune.fit_transform(feat_tune)
X_tune, y_tune = create_dataset(feat_tune_scaled, TIME_STEP, NUM_FEATURES)
print("Bentuk data tuning:", X_tune.shape)

tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=15,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='cnn_walkforward_tuning',
    overwrite=True,
)

print("Menjalankan hyperparameter search untuk 1D-CNN...")
tuner.search(X_tune, y_tune, epochs=15, batch_size=16, validation_split=0.2, verbose=0)
print("Selesai.")

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
best_hp_values = dict(best_hps.values)
print("\n=== Hyperparameter 1D-CNN terbaik ===")
for k, v in best_hp_values.items():
    print(f"{k}: {v}")

# --- catat SEMUA trial (untuk dilampirkan di laporan) ---
trial_records = []
for trial_id, trial in tuner.oracle.trials.items():
    rec = dict(trial.hyperparameters.values)
    rec['trial_id'] = trial_id
    rec['val_loss'] = trial.score
    trial_records.append(rec)

cnn_tuning_results = pd.DataFrame(trial_records).sort_values('val_loss').reset_index(drop=True)
cnn_tuning_results.insert(0, 'rank', range(1, len(cnn_tuning_results) + 1))

print("\n=== Log Lengkap Hyperparameter Tuning 1D-CNN (semua trial) ===")
display(cnn_tuning_results)

cnn_tuning_results.to_csv('cnn_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: cnn_hyperparameter_tuning_log.csv")


## 8. Tuning Hyperparameter — ARIMA (Grid Search Orde p, d, q)

Untuk ARIMA, "hyperparameter" yang di-tuning adalah orde `(p, d, q)` dengan batas `p <= 2, d <= 1, q <= 2`.
Grid search berbasis **AIC** dilakukan **per deret** (per kombinasi *state* x *kategori*) karena karakteristik
tiap deret berbeda. Semua kombinasi yang dicoba beserta AIC-nya dicatat di `arima_tuning_results_all`,
sedangkan orde terbaik per deret ada di `arima_best_orders`.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA


def arima_order_search(train_series, p_max=2, d_max=1, q_max=2):
    """Grid search orde ARIMA berbasis AIC pada data latih awal (sebelum periode uji walk-forward)."""
    best_aic = np.inf
    best_order = (0, 1, 0)
    records = []
    for p, d, q in itertools.product(range(p_max + 1), range(d_max + 1), range(q_max + 1)):
        try:
            fit = ARIMA(train_series, order=(p, d, q)).fit()
            records.append({'p': p, 'd': d, 'q': q, 'AIC': fit.aic})
            if fit.aic < best_aic:
                best_aic = fit.aic
                best_order = (p, d, q)
        except Exception:
            continue
    return best_order, best_aic, pd.DataFrame(records)


arima_best_orders = []
arima_all_trials = []

for cat in CATEGORIES:
    for state in STATES:
        series_df = weekly_avg_enriched[
            (weekly_avg_enriched['cat_id'] == cat) &
            (weekly_avg_enriched['state_id'] == state)
        ].sort_values('wm_yr_wk')

        if len(series_df) < TIME_STEP + 20:
            print(f"Lewati {cat}-{state}: data terlalu pendek ({len(series_df)} minggu)")
            continue

        n_test = min(TEST_WEEKS, len(series_df) - TIME_STEP - 10)
        test_start = len(series_df) - n_test
        initial_train_rev = series_df['revenue'].values[:test_start]

        order, aic, trials_df = arima_order_search(initial_train_rev)
        trials_df['cat_id'] = cat
        trials_df['state_id'] = state
        arima_all_trials.append(trials_df)

        arima_best_orders.append({
            'cat_id': cat, 'state_id': state,
            'best_p': order[0], 'best_d': order[1], 'best_q': order[2],
            'AIC': aic, 'n_test_weeks': n_test
        })
        print(f"{cat:10s} {state} -> orde terbaik ARIMA(p,d,q)={order}, AIC={aic:.2f}")

arima_best_orders = pd.DataFrame(arima_best_orders)
arima_tuning_results_all = pd.concat(arima_all_trials, ignore_index=True)

print("\n=== Orde ARIMA Terbaik per Deret ===")
display(arima_best_orders)

print("\n=== Log Lengkap Grid Search ARIMA (semua kombinasi p,d,q per deret) ===")
display(arima_tuning_results_all.sort_values(['cat_id', 'state_id', 'AIC']))

arima_best_orders.to_csv('arima_best_orders.csv', index=False)
arima_tuning_results_all.to_csv('arima_hyperparameter_tuning_log.csv', index=False)
print("Disimpan ke: arima_best_orders.csv dan arima_hyperparameter_tuning_log.csv")


## 9. Evaluasi Walk-Forward yang Adil (1D-CNN vs ARIMA)

**Protokol (identik untuk kedua model):**
1. Setiap deret mingguan dibagi: `TEST_WEEKS` minggu terakhir jadi periode uji *walk-forward*; sisanya jadi data latih awal.
2. Di setiap langkah minggu uji ke-*t*, kedua model **di-retrain ulang** memakai seluruh histori yang tersedia sampai minggu *t-1* (*expanding window*), lalu memprediksi 1 minggu ke depan (h=1):
   - **ARIMA** memakai orde `(p, d, q)` hasil tuning di atas.
   - **1D-CNN** memakai arsitektur & hyperparameter hasil tuning Keras Tuner, input berupa jendela `TIME_STEP` minggu terakhir.
3. Kedua model memprediksi **minggu target yang sama persis**, dari histori yang tersedia sama persis di setiap langkah -> hasilnya *apple-to-apple*.
4. Skala penyebut RMSSE (RMS dari selisih pertama) dihitung dari **10 minggu terakhir sebelum periode uji dimulai**, identik untuk kedua model.

Metrik yang dihitung: **RMSSE, MAE, RMSE, MAPE**.


In [ ]:
def build_cnn_from_hp(hp_values, time_step, num_features):
    """Bangun model CNN dari hyperparameter tetap hasil tuning."""
    inputs = Input(shape=(time_step, num_features))
    x = inputs
    for i in range(hp_values['num_cnn_layers']):
        x = Conv1D(
            filters=hp_values[f'filters_{i}'],
            kernel_size=hp_values[f'kernel_size_{i}'],
            activation='relu',
            padding='causal',
        )(x)
    x = Flatten()(x)
    outputs = Dense(1)(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(hp_values['learning_rate']))
    return model


def walk_forward_cnn(feature_matrix, time_step, n_test, hp_values, epochs=15, batch_size=8):
    """CNN di-retrain ulang (expanding window) di setiap langkah walk-forward."""
    n = len(feature_matrix)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        train_data = feature_matrix[:i]
        X_train, y_train = create_dataset(train_data, time_step, feature_matrix.shape[1])
        if len(X_train) < 5:
            preds.append(train_data[-1, 0])
            continue
        model = build_cnn_from_hp(hp_values, time_step, feature_matrix.shape[1])
        model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)
        X_input = feature_matrix[i - time_step:i].reshape(1, time_step, feature_matrix.shape[1])
        pred = model.predict(X_input, verbose=0)[0, 0]
        preds.append(pred)
    return np.array(preds)


def walk_forward_arima(series_1d, order, n_test):
    """ARIMA di-fit ulang (expanding window) di setiap langkah walk-forward, orde tetap."""
    n = len(series_1d)
    test_start = n - n_test
    preds = []
    for i in range(test_start, n):
        hist = series_1d[:i]
        try:
            fc = ARIMA(hist, order=order).fit().forecast(steps=1)[0]
        except Exception:
            fc = hist[-1]
        preds.append(fc)
    return np.array(preds)


print("Fungsi walk-forward CNN & ARIMA siap dipakai.")


In [ ]:
walkforward_results = []
walkforward_predictions = {}  # simpan untuk plotting ulang jika perlu

for cat in CATEGORIES:
    for state in STATES:
        series_df = weekly_avg_enriched[
            (weekly_avg_enriched['cat_id'] == cat) &
            (weekly_avg_enriched['state_id'] == state)
        ].sort_values('wm_yr_wk').reset_index(drop=True)

        if len(series_df) < TIME_STEP + 20:
            print(f"Lewati {cat}-{state}: data terlalu pendek")
            continue

        n_test = min(TEST_WEEKS, len(series_df) - TIME_STEP - 10)
        test_start = len(series_df) - n_test

        order_row = arima_best_orders[
            (arima_best_orders['cat_id'] == cat) & (arima_best_orders['state_id'] == state)
        ]
        if order_row.empty:
            print(f"Lewati {cat}-{state}: orde ARIMA belum di-tuning")
            continue
        order = (int(order_row['best_p'].iloc[0]), int(order_row['best_d'].iloc[0]), int(order_row['best_q'].iloc[0]))

        # --- fitur untuk CNN (revenue, wday, month), di-scale ---
        feat = series_df[['revenue', 'wday', 'month']].values
        scaler = MinMaxScaler()
        feat_scaled = scaler.fit_transform(feat)

        print(f"\n>>> {cat} - {state} | n_test={n_test} minggu | orde ARIMA={order}")

        cnn_pred_scaled = walk_forward_cnn(
            feat_scaled, TIME_STEP, n_test, best_hp_values,
            epochs=CNN_EPOCHS, batch_size=CNN_BATCH
        )
        dummy = np.zeros((len(cnn_pred_scaled), NUM_FEATURES))
        dummy[:, 0] = cnn_pred_scaled
        cnn_pred = scaler.inverse_transform(dummy)[:, 0]

        arima_pred = walk_forward_arima(series_df['revenue'].values, order, n_test)

        actual = series_df['revenue'].values[test_start:]
        scale_hist = series_df['revenue'].values[test_start - TIME_STEP: test_start]

        walkforward_predictions[(cat, state)] = {
            'actual': actual, 'cnn': cnn_pred, 'arima': arima_pred
        }

        for model_name, pred in [('1D-CNN', cnn_pred), ('ARIMA', arima_pred)]:
            mae = mean_absolute_error(actual, pred)
            rmse = np.sqrt(mean_squared_error(actual, pred))
            mape = np.mean(np.abs((actual - pred) / (actual + 1e-8))) * 100
            rmsse = wrmsse_score(actual, pred, scale_hist)

            walkforward_results.append({
                'Category': cat, 'State': state, 'Model': model_name,
                'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'RMSSE': rmsse
            })
            print(f"  {model_name:7s} -> MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%  RMSSE={rmsse:.4f}")

        # --- plot actual vs CNN vs ARIMA ---
        plt.figure(figsize=(10, 4))
        plt.plot(actual, label='Actual', linewidth=2)
        plt.plot(cnn_pred, '--', label='1D-CNN (walk-forward)')
        plt.plot(arima_pred, ':', label='ARIMA (walk-forward)')
        plt.title(f"Walk-Forward Forecast: {cat} - {state}")
        plt.xlabel("Test Week")
        plt.ylabel("Revenue")
        plt.legend()
        plt.tight_layout()
        plt.show()

walkforward_results_df = pd.DataFrame(walkforward_results)
walkforward_results_df.to_csv('walkforward_cnn_vs_arima_results.csv', index=False)
print("\nHasil lengkap disimpan ke: walkforward_cnn_vs_arima_results.csv")
display(walkforward_results_df)


## 10. Tabel Perbandingan Akhir

Ringkasan performa 1D-CNN vs ARIMA per metrik, dipecah menurut *State* x *Kategori*, ditutup dengan rata-rata keseluruhan per model.

In [ ]:
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

for metric in ['RMSSE', 'MAE', 'RMSE', 'MAPE']:
    pivot = walkforward_results_df.pivot_table(
        index='State', columns=['Model', 'Category'], values=metric
    )
    pivot = pivot.reindex(columns=pd.MultiIndex.from_product(
        [['1D-CNN', 'ARIMA'], ['HOBBIES', 'HOUSEHOLD', 'FOODS']]
    ))
    print(f"\n=== Tabel {metric} -- 1D-CNN vs ARIMA (Walk-Forward, Adil) ===")
    display(pivot)

summary = walkforward_results_df.groupby('Model')[['MAE', 'RMSE', 'MAPE', 'RMSSE']].mean().round(4)
print("\n=== Ringkasan Rata-Rata Keseluruhan (semakin kecil semakin baik) ===")
display(summary)


## 11. Kesimpulan

Bandingkan baris **Ringkasan Rata-Rata Keseluruhan** di atas: model dengan nilai **RMSSE / MAE / RMSE / MAPE**
yang lebih kecil secara konsisten di sebagian besar kombinasi *state* x *kategori* adalah model yang lebih baik
pada protokol walk-forward ini. Perhatikan juga tabel per kategori — performa relatif kedua model bisa berbeda
antar kategori (mis. 1D-CNN mungkin lebih unggul pada deret yang lebih bervariasi, sementara ARIMA lebih stabil
pada deret yang lebih halus/musiman).